# B4: How does the gain from decision features change with training-set size?

Training sizes 128 to 16,384 rows: nested stratified subsamples of each outer-training fold (at least one example per class; classes are never dropped), test rows unchanged; 5 draws up to 512 rows, 3 at 1,024–2,048, 1 above. Main curve: all 18 datasets at 128–2,048; extended curve: the seven datasets with at least 16,384 training rows in every fold, at every size. Arms: baseline, each decision model's both view, frozen embeddings (per-column PCA32, three encoders) and the E5-small per-column label-trained classifier; PCA and classifier fit on the sampled rows only. This deliberately subsamples training rows: an explicit, recorded, owner-approved design, not a silent cap.

Seven predictors use prespecified settings on the same five outer folds across 18 datasets (3 binary, 6 multiclass, 9 regression). They are linear/logistic regression, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2 and Kumo. Each structured baseline includes the non-text columns and one text-availability flag per text column. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: each contributes its five-fold mean paired gain; raw gains stay within a task family (ΔAUC and ΔR², N = 9 each, separate panels) and only the headroom share (gain / (1 − baseline)) pools all 18 datasets. Decision medians include every member of the recorded decision-model cohort. At each size, the table reports the pooling method and dataset count.

Reads `output_JEVPFN/B/exp09_data_efficiency/results/group_b/b4_data_efficiency.<cohort>-<roster digest>.zip`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, group_b, group_b_views as views, learner_scores, study_export
from src.visualize import critical_difference, figures, style
from src.visualize import group_b as gbfig
from src.evaluation import data_efficiency
style.apply()
NOTEBOOK = "B/exp09_data_efficiency"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete: export, provenance and coverage")
display_json("Notebook execution identity", provenance())
frames, receipt = benchmark_export.read("B4")
EXPORTS = {"B4": (frames, receipt)}
ROSTER = benchmark_export.decision_roster()
display_json("Decision-model cohort and members", benchmark_export.cohort_record())
WAITING = "Waiting for export: no measurements are shown."
for study, (frames_, receipt_) in EXPORTS.items():
    print(f"{study}: {study_export.describe(receipt_)}")
    record({k: receipt_.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, f"{study} export identity")
    if frames_:
        display_table(f"{study}: planned versus complete five-fold arms per phase and predictor", frames_["coverage"], preview_rows=14)
        if "median_roster_gaps" in frames_ and not frames_["median_roster_gaps"].empty:
            display_table(f"{study}: dataset x predictor cells whose median lacks configured decision models", frames_["median_roster_gaps"])

B4: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp09_data_efficiency\results\group_b\b4_data_efficiency.open_weight-5baeccd37d70d034.zip is absent. No measurements are shown.


## 2. Summary: gain against labelled rows (main curve)

Decision both (median model), E5-small frozen and the label-trained classifier; mean over the seven predictors with the predictor range as a faint band.

The curve measures gain at equal label count. Repeated draws reuse the same outer test rows. When every dataset has a positive variance estimate across draws, the table reports a DerSimonian–Laird interval. For one draw or zero draw variance it reports an equal-weight mean and an approximate normal interval from the spread across datasets. At the largest sizes the raw-gain estimate uses four classification datasets or three regression datasets.

In [2]:
report_section("2. Summary: gain against labelled rows")
if not frames:
    print(WAITING)
else:
    head = views.fold_long(frames["folds"])
    curves = views.size_curves(head, roster=ROSTER, curves=data_efficiency.config()["curves"], representations=gbfig.SIZE_ARMS)
    gbfig.render(save, gbfig.size_curves(curves, name="b4_main_curve", what="Amount of labelled data"))
    display_table("Pooled gain per task family, curve, arm, predictor and size", curves, preview_rows=20)

Waiting for export: no measurements are shown.


## 3. Supporting views: per predictor, extended curve and slopes per doubling

In [3]:
report_section("3. Per predictor, extended curve and slopes")
if not frames:
    print(WAITING)
else:
    for family in style.TASK_FAMILIES:
        gbfig.render(save, gbfig.size_curves_by_predictor(curves, name=f"b4_per_predictor_{family}", what="Amount of labelled data", family=family))
    gbfig.render(save, gbfig.size_curves(curves, name="b4_extended_curve", what="Amount of labelled data", curve="extended"))
    slopes = views.size_slopes(head, roster=ROSTER, representations=gbfig.SIZE_ARMS)
    gbfig.render(save, gbfig.size_slopes(slopes, name="b4_slopes_per_doubling", what="Amount of labelled data"))
    display_table("Per-dataset slope per doubling (median over predictors, range)", slopes, preview_rows=18)
    display_table("Raw gain by task family and training size, with pooling method", frames["pooled_raw"], preview_rows=10)

Waiting for export: no measurements are shown.


## 4. Statistics

Each exploratory comparison uses the predictor’s baseline at the same training size. Holm adjustment is applied separately within each predictor and size.

In [4]:
report_section("4. Statistics")
framework = learner_scores.run_application("B4_data_efficiency")
print(f"B4_data_efficiency: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application B4_data_efficiency", learner_scores.summary_record(framework))
gbfig.render(save, critical_difference.cd_figures(framework))

B4_data_efficiency: awaiting_export (framework.status = approved)


In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified exports and local caches named above were read; no inference, model training or VSC changes were performed.")

B/exp09_data_efficiency

1. What is complete: export, provenance and coverage


Notebook execution identity
{
  "code_sha256": "8b4d9540fb90939d06c1177e768d032c9866801857bd06cfd36cb4fa3aea5040",
  "git_head": "e5d935fe73a631187c6d40435fbeb13c6743c494",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Decision-model cohort and members
{
  "decision_cohort": "open_weight",
  "decision_roster": [
    "liquid",
    "liquid_lfm3",
    "perplexity",
    "cloudflare",
    "strands",
    "intern",
    "h2o",
    "sii",
    "kev",
    "decision3",
    "nimble",
    "glide",
    "mammoth",
    "drex_max",